In [1]:
!nvidia-smi

Tue Mar 31 19:49:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/Nusultan11/imdb-sentiment.git
%cd /content/imdb-sentiment

Cloning into 'imdb-sentiment'...
remote: Enumerating objects: 607, done.
remote: Counting objects: 100% (607/607), done.
remote: Compressing objects: 100% (347/347), done.
remote: Total 607 (delta 324), reused 476 (delta 193), pack-reused 0 (from 0)
Receiving objects: 100% (607/607), 549.98 KiB | 20.37 MiB/s, done.
Resolving deltas: 100% (324/324), done.
/content/imdb-sentiment


In [3]:
!python -m pip install --upgrade pip
!pip install -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Obtaining file:///content/imdb-sentiment
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for imdb-sentiment (pyproject.toml) ... done
  Created wheel for imdb-sentiment: filename=imdb_sentiment-0.1.0-0.editable-py3-none-any.whl size=5679 sha256=cee4f25262b8856b658859ec8f718009b653f0d9a2d97b2be9cdd08ccff6e584
  Stored in directory: /tmp/pip-ephem-wheel-cache-w2rx2rgm/wheels/3e/08/09/1f8f2dc306b20c0f6d4e5a4ddb080c355c7ce988d70820356f
Successfully built imdb-sentiment


In [13]:
import os
import sys
from pathlib import Path
import json
import zipfile

repo_root = Path("/content/imdb-sentiment")
os.chdir(repo_root)

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("repo_root =", repo_root)
print("src_path =", src_path)

from imdb_sentiment.settings import load_config

from google.colab import files

repo_root = /content/imdb-sentiment
src_path = /content/imdb-sentiment/src


In [5]:
config = load_config("configs/experiments/lstm_bidirectional_masked_mean_optuna_regexprep_v1.yaml")

print("experiment:", config.experiment.name)
print("family:", config.experiment.family)
print("bidirectional:", config.model.bidirectional)
print("pooling:", config.model.pooling)
print("preprocessing:", config.model.preprocessing)
print("batch_size:", config.model.batch_size)
print("epochs:", config.model.epochs)
print("dropout:", config.model.dropout)
print("lr:", config.model.lr)
print("model_output:", config.paths.model_output)

experiment: bidirectional_masked_mean_optuna_regexprep_v1
family: lstm
bidirectional: True
pooling: masked_mean
preprocessing: regex_v2
batch_size: 16
epochs: 5
dropout: 0.5
lr: 0.002074566765675252
model_output: /content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_masked_mean_optuna_regexprep_v1/model.pt


In [6]:
os.environ["PYTHONPATH"] = "src"

!python -m imdb_sentiment.cli train --config configs/experiments/lstm_bidirectional_masked_mean_optuna_regexprep_v1.yaml

README.md: 7.81kB [00:00, 21.6MB/s]
plain_text/train-00000-of-00001.parquet: 100% 21.0M/21.0M [00:01<00:00, 17.1MB/s]
plain_text/test-00000-of-00001.parquet: 100% 20.5M/20.5M [00:00<00:00, 33.5MB/s]
plain_text/unsupervised-00000-of-00001.p(…): 100% 42.0M/42.0M [00:00<00:00, 51.8MB/s]
Generating train split: 100% 25000/25000 [00:00<00:00, 102784.24 examples/s]
Generating test split: 100% 25000/25000 [00:00<00:00, 101457.75 examples/s]
Generating unsupervised split: 100% 50000/50000 [00:00<00:00, 107984.79 examples/s]
Training finished.
Accuracy: 0.9046
Precision: 0.8844
Recall: 0.9314
F1: 0.9073
Model saved to: /content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_masked_mean_optuna_regexprep_v1/model.pt
Validation metrics saved to: /content/imdb-sentiment/artifacts/experiments/lstm/bidirectional_masked_mean_optuna_regexprep_v1/val_metrics.json


In [14]:
artifact_dir = Path("artifacts/experiments/lstm/bidirectional_masked_mean_optuna_regexprep_v1")
zip_path = artifact_dir / "regex_v2_bilstm_masked_pooling_trained_bundle.zip"

files_to_pack = [
    "model.pt",
    "vocab.json",
    "training_config.json",
    "training_history.json",
    "threshold_tuning.json",
    "val_metrics.json",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for filename in files_to_pack:
        file_path = artifact_dir / filename
        if file_path.exists():
            zf.write(file_path, arcname=filename)

print("bundle:", zip_path)
print("exists:", zip_path.exists())
print("size_bytes:", zip_path.stat().st_size if zip_path.exists() else None)

bundle: artifacts/experiments/lstm/bidirectional_masked_mean_optuna_regexprep_v1/regex_v2_bilstm_masked_pooling_trained_bundle.zip
exists: True
size_bytes: 15694221


In [15]:
files.download("artifacts/experiments/lstm/bidirectional_masked_mean_optuna_regexprep_v1/regex_v2_bilstm_masked_pooling_trained_bundle.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>